# 02 Introduction to Data Wrangling: From CSV to an Analysis-Ready DataFrame

Pine Cedar Nursing Home Legionnaires' disease outbreak—the 280-row line list has been compiled.
In this lesson we read the CSV into pandas, check its quality, build derived variables, and produce an attack-rate table by wing.

## What Is a DataFrame?

**pandas** is Python's most widely used data-processing package; you can think of it as "Excel for Python."

- **DataFrame** = a two-dimensional table (rows × columns), just like an Excel worksheet
- **Series** = a one-dimensional column, just like a single column in Excel

Every operation in this lesson revolves around the DataFrame.

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: Read in the line list ---
# import pandas as pd → import pandas under the nickname pd (a worldwide convention)
# pd.read_csv() → read a CSV file and return a DataFrame
# df.shape → (n_rows, n_cols), [0] for rows, [1] for columns
# df.head() → show the first 5 rows (you can pass a number like df.head(10))

import pandas as pd

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
print(f"Data dimensions: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# --- Step 2: Inspect the data structure ---
# df.info() tells you:
#   - the "name" and "type" of each column
#     (int64=integer, float64=float, object=text, bool=boolean)
#   - how many "non-null values" each column has → below 280 means there's missing data
# 💡 the object type usually means text; dates are read in as object too and need converting

df.info()

In [ ]:
# df.describe() gives you a statistical summary of each numeric column:
#   count=non-null count, mean=average, std=standard deviation
#   min/max=minimum and maximum, 25%/50%/75%=quartiles
# Key: are the age min/max reasonable? Any anomalies like -1 or 999?

df.describe()

In [ ]:
# --- Step 3: Date conversion ---
# Dates read in from the CSV are "text" (object); Python doesn't know they're dates
# You must use pd.to_datetime() to convert them to the datetime type before you can
# sort, subtract, extract months, etc.
#
# pd.to_datetime(df[col]) → text "2026-01-15" becomes a datetime object
# errors="coerce" → don't raise on blanks or "N/A"; turn them into NaT (Not a Time, the date version of a missing value)
# df[col] = ... → store the converted result back into the original column

date_cols = [
    "facility_admission_date",
    "symptom_onset_date",
    "hospitalization_date",
    "death_date",
    "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Verify the conversion: these should all be datetime64[ns] now, no longer object
df[date_cols].dtypes

In [ ]:
# --- Step 4: Build derived variables ---
# Syntax: df["new_column_name"] = formula (same idea as adding a column in Excel)

# 1) Age group — pd.cut() bins continuous numbers (like grading exams A/B/C/D)
#    bins=[59,69,79,89,100] are the cut points (left-open, right-closed)
#    (59,69]=age 60-69, (69,79]=age 70-79, ...
df["age_group"] = pd.cut(
    df["age"],
    bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# 2) Comorbidity count — .sum(axis=1) is "summing across columns" (for each person, sum their 5 comorbidity columns)
#    axis=0 = downward (average of each column); axis=1 = rightward (sum of each row)
comorbidity_cols = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd",
    "immunosuppressed",
]
df["n_comorbidities"] = df[comorbidity_cols].sum(axis=1)

# 3) Infected or not — boolean operation (!= "not_ill" → True/False) then convert to 0/1
#    .astype(int) turns True→1, False→0
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 4) Days from onset to hospitalization — subtract two dates to get a time difference, .dt.days gets the days
#    .dt is the "datetime accessor": .dt.year .dt.month .dt.days (day difference)
df["onset_to_hosp_days"] = (
    df["hospitalization_date"] - df["symptom_onset_date"]
).dt.days

# 5) Epi week — the ISO 8601 standard week number (1~53), often used for "weekly statistics" in outbreaks
df["epi_week"] = df["symptom_onset_date"].dt.isocalendar().week

# Inspect the new columns
df[["case_id", "age", "age_group", "n_comorbidities", "infected",
    "onset_to_hosp_days", "epi_week"]].head(10)

In [ ]:
# --- Step 5: Handle missing values ---
# Missing values = the "blanks" in your data; pandas uses three symbols for them:
#   NaN (Not a Number) → missing in a numeric column
#   NaT (Not a Time)   → missing in a date column
#   None               → Python's native "no value"
#
# df.isnull() → returns True/False for each cell (whether it's missing)
# .sum()      → sums the Trues in each column (True=1, False=0)
# missing[missing > 0] → show only columns that have missing values (boolean filter)
#
# ⚠️ "Structural missingness" vs "data error":
#   - a non-infected person has no onset date → structural missingness (normal, no need to fill)
#   - an infected person's age field is blank → a data error (needs to be re-entered)

print("=== Missing-value count per column ===")
missing = df.isnull().sum()
print(missing[missing > 0].to_string())

# df.loc[condition, column_name] → first filter rows (condition), then take a specific column
# .notna() → the opposite of .isnull(); True means "has a value"
# .sum()   → count how many have a value
print(f"\nNumber of non-infected people with an onset date: "
      f"{df.loc[df['infected'] == 0, 'symptom_onset_date'].notna().sum()}")
print("→ 0 means the structural missingness is fine")

In [ ]:
# --- Step 6: Grouped statistics with groupby ---
# groupby is one of pandas's most powerful features; the concept is like Excel's "Pivot Table":
#   1) Split: cut the data into small groups by floor × wing
#   2) Apply: compute a statistic for each group
#   3) Combine: stitch the results back into one table
#
# df.groupby(["floor", "wing"]) → group by floor and wing
# .agg(                         → apply multiple statistics to each group
#     residents=("case_id", "size"),   → count the rows in each group (= number of residents)
#     infected=("infected", "sum"),    → sum the infected column (1+1+0+... = number infected)
# )
# .reset_index() → "flatten" the multi-level index groupby produces back into ordinary columns
#
# ⚠️ Attack rate = number infected ÷ number of residents in that area (not divided by all 280 people!)
#    Getting the denominator wrong is one of the most common mistakes in outbreak reports

wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate"] = wing_stats["infected"] / wing_stats["residents"]
wing_stats["attack_rate_pct"] = (wing_stats["attack_rate"] * 100).round(1)

print("=== Attack rate by wing ===")
print(wing_stats.to_string(index=False))

In [ ]:
# --- Step 6b: Advanced data operations ---
# Below are the advanced techniques Excel users use most when moving to pandas

# === Frequency table: value_counts() ===
# First thing to do with data: look at the frequency distribution of each column (like Excel's COUNTIF)
print("=== Clinical severity distribution ===")
print(df["clinical_severity"].value_counts())
print("\nPercentages:")
print((df["clinical_severity"].value_counts(normalize=True) * 100).round(1))

# === Pivot table: pivot_table() ===
# This is Excel's Pivot Table! index=row labels, columns=column labels, values=the value
pivot = pd.pivot_table(
    df,
    values="infected",
    index="wing",
    columns="floor",
    aggfunc="mean",
    margins=True,
    margins_name="Total",
)
print("\n=== Attack rate (%) by wing × floor ===")
print((pivot * 100).round(1))

# === Cross-tabulation: crosstab() ===
# Quickly build a 2×2 table (Ch03 goes deeper)
print("\n=== Sex × infection status ===")
print(pd.crosstab(df["sex"], df["infected"], margins=True))

In [ ]:
# === Method Chaining: write your analysis in one line ===
# The traditional style needs many temporary variables; method chaining links steps into a pipeline
# .query() writes filter conditions as a string (and/or/not instead of &/|/~)
# .assign() adds columns directly in the chain

# Traditional style vs method chaining
# Traditional:
# cases = df[df["infected"] == 1]
# elderly = cases[cases["age"] >= 80]
# result = elderly.groupby("floor").size().reset_index(name="n")

# Method chaining (all in one go):
result = (
    df
    .query("infected == 1 and age >= 80")
    .groupby("floor")
    .size()
    .reset_index(name="n_elderly_cases")
    .sort_values("n_elderly_cases", ascending=False)
)
print("=== Infected people aged 80+, by floor ===")
print(result.to_string(index=False))

# A more complex chaining example: compute attack rate and case fatality rate at once
summary = (
    df
    .assign(dead=(df["outcome"] == "dead").astype(int))
    .groupby("floor")
    .agg(
        n=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("dead", "sum"),
    )
    .assign(
        attack_rate=lambda d: (d["infected"] / d["n"] * 100).round(1),
        cfr=lambda d: (d["deaths"] / d["infected"] * 100).round(1),
    )
)
print("\n=== Attack rate + case fatality rate by floor ===")
print(summary)

In [ ]:
# === merge: joining tables (the VLOOKUP equivalent) ===
# Outbreak work often needs to combine data from different sources (line list + lab results)
# pd.merge(left, right, on=common_column, how=join_type)
# how="left" → keep all rows from the left table (most common, doesn't drop cases)

# Simulate a lab-results table
lab_df = df[df["lab_confirmed"] == True][["case_id"]].head(10).copy()
lab_df["ct_value"] = [25.3, 28.1, 22.5, 31.0, 24.8, 27.2, 23.1, 29.5, 26.0, 30.2]

# Merge onto the first 20 residents
sample = df[["case_id", "age", "sex", "infected"]].head(20)
merged = pd.merge(sample, lab_df, on="case_id", how="left")
print("=== merge result (first 20 residents + lab ct_value) ===")
print(merged.to_string(index=False))
print(f"\nBefore merge: {len(sample)} rows → after merge: {len(merged)} rows (how='left' loses nothing)")

# === .str string operations + drop_duplicates ===
# The .str accessor lets you operate on a whole text column (no loop needed)
print("\n=== Wing column standardized to uppercase ===")
print(df["wing"].str.upper().value_counts())

# drop_duplicates: remove duplicates (duplicate reports are common in outbreak work)
print(f"\nBefore dedup: {len(df)} rows")
df_unique = df.drop_duplicates(subset="case_id", keep="first")
print(f"After dedup: {len(df_unique)} rows")

# nlargest: quickly find the top N
print("\n=== Top 3 wings by attack rate ===")
print(wing_stats.nlargest(3, "attack_rate_pct")[["floor", "wing", "attack_rate_pct"]].to_string(index=False))

## Summary

In this lesson you completed the full line-list data-wrangling workflow:

**The six basic steps**
1. **Read in** the CSV → `pd.read_csv()`
2. **Inspect** the structure → `df.info()`, `df.describe()`
3. **Convert** dates → `pd.to_datetime()`
4. **Derive** new variables → `pd.cut()`, `.sum(axis=1)`, `.dt.days`
5. **Verify** missing values → structural missingness vs data error
6. **Compute** grouped metrics → `groupby().agg()`

**Advanced operations (Step 6b)**
- **Frequency tables** → `value_counts()`, `pd.crosstab()`
- **Pivot tables** → `pd.pivot_table()` (the Excel Pivot Table equivalent)
- **Method chaining** → the `.query().groupby().assign()` pipeline
- **Joining tables** → `pd.merge()` (the VLOOKUP equivalent)
- **Text cleaning** → `.str.upper()`, `.str.contains()`, `drop_duplicates()`

In the next lesson we'll use this cleaned data to make charts—epidemic curves, age distributions, wing comparisons, heatmaps, and interactive charts.

> 📄 **pandas cheat sheet**: [Pandas Cheat Sheet (PDF)](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)